In [1]:
import uuid
from typing import Dict, Any, Optional
from google.adk.agents import Agent
from google.adk.events import Event
from google.adk.models.lite_llm import LiteLlm
from google.adk.tools import FunctionTool
from google.adk.runners import InMemoryRunner
from google.genai import types

In [2]:
# If using LiteLLM as translation layer to call Claude
model = LiteLlm(model="anthropic/claude-haiku-4-5-20251001")

# If using Gemini model through Google AI Studio API Quota
# model = "gemini-2.0-flash"

In [3]:
# Define tool functions simulating the actions of the specialist agents.

def booking_handler(request: str) -> str:
    """
    Handles booking requests for flights and hotels.
    Args:
        request: The user's request for a booking.
    Returns:
        A confermation messege that the booking was handled.
    """
    print("--- Booking Handler Called ---")
    return f"Booking action for '{request}' has been simulated."

def info_handler(request: str) -> str:
    """
    Handles general information requests.
    Args:
        request: The user's question.
    Returns:
        A message indicating the information request was handled.
    """
    print("--- Info Handler Called ---")
    return f"Information request for '{request}'. Information retrieval has been simulated."

In [4]:
# Create tools from functions

booking_tool = FunctionTool(booking_handler)
info_tool = FunctionTool(info_handler)

In [5]:
# Define specialized sub-agents equipped with their respective tools

booking_agent = Agent(
    name = "Booker",
    model = model,
    description = "A specialized agent that handles all flight and hotel booking requests by calling the booking tool.",
    # Detailed instructions are needed when using models different from Gemini
    # Whitout explicit instructions Claude models fail to call tools
    instruction=(
        "You are a booking specialist. When you receive a booking request, "
        "you MUST immediately call the booking_handler tool with the user's request. "
        "Do NOT ask follow-up questions. "
        "After receiving the tool result, return it exactly as-is. "
        "Do NOT add any extra commentary, suggestions, or follow-up questions."
    ),
    tools = [booking_tool]
)

info_agent = Agent(
    name = "Info",
    model = model,
    description = "A specialized agent that provides general information and answers user questions by calling the info tool.",
    # Detailed instructions are needed when using models different from Gemini
    # Whitout explicit instructions Claude models fail to call tools
    instruction=(
        "You are an information specialist. When you receive a question, "
        "you MUST immediately call the info_handler tool with the user's request. "
        "Do NOT answer directly. "
        "After receiving the tool result, return it exactly as-is. "
        "Do NOT add any extra commentary, suggestions, or follow-up questions."
    ),
    tools = [info_tool]
)

In [6]:
# Define the parent agent with explicit delegation instructions

coordinator = Agent(
    name = "Coordinator",
    model = model,
    instruction = (
        "You are the main coordinator. Your only taks is to analyze incoming user requests and delagate them to the appropriate "
        "specialist agent. Do not try to the user directly.\n"
        "- For any request related to booking flights or hotels, delegate to the 'Booker' agent.\n"
        "- For all other general information questions, delegate to the 'Info' agent."
    ),
    description = "A coordinator that routes user requests to the correct specialist agent.",
    # The presence of sub_agents enables LLM-driven delegation (Auto-Flow) by default
    sub_agents = [booking_agent, info_agent]
)

In [7]:
# Execution Logic

async def run_coordinator(runner: InMemoryRunner, request: str):
    """
    Runs the coordinator agent with a given request and delegates.
    """
    print(f"--- Running Coordinator agent with request: '{request}' ---")
    final_result = ""
    
    try:
        user_id = "user_123"
        session_id = str(uuid.uuid4())
        
        await runner.session_service.create_session(
            app_name = runner.app_name,
            user_id = user_id,
            session_id = session_id
        )

        for event in runner.run(
            user_id = user_id,
            session_id = session_id,
            new_message = types.Content(
                role = 'user',
                parts = [types.Part(text=request)]
            )
        ):
            if event.is_final_response() and event.content:
                if hasattr(event.content, 'text') and event.content.text:
                    final_result = event.content.text
                elif event.content.parts:
                    text_parts = [part.text for part in event.content.parts if part.text]
                    final_result = "".join(text_parts)
                break

        print(f"Coordinator Final Response: {final_result}")
        return final_result
        
    except Exception as e:

        print(f"An error occurred while processing your request: {e}")
        return f"An error occurred while processing your request: {e}"


async def execution_example(request: str):
    print("--- Google ADK Routing Example ---")
    
    runner = InMemoryRunner(coordinator)
    result = await run_coordinator(runner, request)
    #print(f"\n\nFinal Output: {result}")

In [8]:
await execution_example("Book me an hotel in Melbourne.")

--- Google ADK Routing Example ---
--- Running Coordinator agent with request: 'Book me an hotel in Melbourne.' ---
--- Booking Handler Called ---
Coordinator Final Response: Booking action for 'Book me a hotel in Melbourne' has been simulated.


In [10]:
await execution_example("What is the highest mountain in China?")

--- Google ADK Routing Example ---
--- Running Coordinator agent with request: 'What is the highest mountain in China?' ---
--- Info Handler Called ---
Coordinator Final Response: Information request for 'What is the highest mountain in China?'. Information retrieval has been simulated.


In [15]:
# Mixed behaviour: does not always call the appropriate tool.
await execution_example("Find me a flight for Tokyo for next month.")

--- Google ADK Routing Example ---
--- Running Coordinator agent with request: 'Find me a flight for Tokyo for next month.' ---
--- Booking Handler Called ---
Coordinator Final Response: Booking action for 'Find me a flight for Tokyo for next month.' has been simulated.
